# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

from dotenv import load_dotenv
from typing import List, Dict
import chromadb

from tavily import TavilyClient

In [3]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")

In [5]:
@tool

def retrieve_game(query: str, n:int=1) -> List[Dict]:

    results = collection.query(
        query_texts=[query],
        n_results=n
    )

    docs = []

    for metadata, document in zip(results["metadatas"][0], results["documents"][0]):
        docs.append({
            "Description": metadata["Description"],
            "Genre": metadata["Genre"],
            "Name": metadata["Name"],
            "Publisher": metadata["Publisher"],
            "YearOfRelease": metadata["YearOfRelease"],
            "Platform": metadata["Platform"],
            "content": document
        })

    return docs

#### Evaluate Retrieval Tool

In [6]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: List[Dict]) -> Dict:
    """
    Tool to evaluate the quality of retrieved documents and check if it is useful to answer the query.

    arguments:
    - question
    - documents retrieved from retrieve_game function

    returns:
    - useful: bool
    - description: str
    """

    if not retrieved_docs:
        return {
            "useful": False,
            "description": "No document retrieved from Vector DB."
        }

    # Text matching - better go lower case for both question and retrieved documents
    question_lowercase = question.lower()
    matches = 0
    useful = 0

    for document in retrieved_docs:
        # Take retrieved document name to lower case
        document_name_lowercase = document["Name"].lower()
        if document_name_lowercase in question_lowercase:
            matches += 1

    if matches > 0:
        useful = matches
        description = "Retrieved documents are useful."
    else:
        description = "Retrieved documents cannot be considered useful."
        
    useful = matches > 0 or len(retrieved_docs) >= 2

    return {
        "useful": useful,
        "description": description
    }

#### Game Web Search Tool

In [7]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def game_web_search(question: str) -> List[Dict]:
    """
    Semantic web search using Tavily.

    args:
    - question: a question about game industry
    """

    response = tavily_client.search(
        query=question,
        max_results=3
    )

    return response.get("results", [])

### Agent

In [8]:
class UdaPlayAgent:
    def __init__(self):
        self.memory = []

    def _answer_from_web(self, question, web_results):
        if not web_results:
            return "No reliable information found on the Web."

        top = web_results[0]
        return f"{top['content']} (Source: {top['url']})"
        
    def _answer_from_rag(self, question, docs):
        doc = docs[0]
        return (
            f"{doc['Name']} was released in {doc['YearOfRelease']} "
            f"for {doc['Platform']}. {doc['Description']}"
        )        

    def answer(self, question: str) -> Dict:

        retrieved_docs = retrieve_game(question)

        evaluation = evaluate_retrieval(question, retrieved_docs)

        if evaluation["useful"]:
            answer = self._answer_from_rag(question, retrieved_docs)
            source = "INTERNAL VECTOR DB"
        else:
            web_search_results = game_web_search(question)
            answer = self._answer_from_web(question, web_search_results)
            source = "FROM WEB"
            self.memory.append(web_search_results)

        return {
            "question": question,
            "answer": answer,
            "source": source,
            "evaluation": evaluation
        }



In [9]:
agent = UdaPlayAgent()

queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
    "When was EA Sports FC 25 released?",
    "Was there any Halo Infinite released in 2021?",
]

for q in queries:
    response = agent.answer(q)
    print("Question:", response["question"])
    print(" "*5, "Answer:", response["answer"])
    print(" "*5, "Source:", response["source"])
    print(" "*5, "Evaluation Useful:", response["evaluation"]['useful'])
    print(" "*5, "Description:", response["evaluation"]['description'])
    print("-"*5)

Question: When Pokémon Gold and Silver was released?
      Answer: Pokémon Gold and Silver was released in 1999 for Game Boy Color. Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
      Source: INTERNAL VECTOR DB
      Evaluation Useful: True
      Description: Retrieved documents are useful.
-----
Question: Which one was the first 3D platformer Mario game?
      Answer: *Super Mario 64* is the first 3D Mario game. But it’s not just that this was Mario’s first foray into 3D — *Super Mario 64* became the benchmark for *all* 3D games in the mid-90s. Being the first 3D Mario game, *Super Mario 64* really set the tone for what was to come. *Super Mario 64* set the standard, and its DNA is present across all 3D Mario games. *Super Mario Sunshine* is regarded — understandably — as the black sheep of the 3D Mario games. Partly because the Wii U sold so poorly, relative to other Nintendo machines, I’d wager that *Super Mario 3D World* is probably the l

### (Optional) Advanced

In [10]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes